# 🧠 Materi Kuliah Deep Learning
## Pertemuan 2: Neural Networks & Backpropagation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hendrick02121977/Deep-Learning/blob/main/notebooks/02_Neural_Networks_dan_Backpropagation.ipynb)

---

**Tujuan Pembelajaran:**
- Memahami proses Forward Propagation secara matematis
- Memahami konsep dan implementasi Backpropagation
- Mengenal berbagai algoritma optimasi (SGD, Adam, RMSprop)
- Memahami konsep Gradient Descent
- Mengimplementasikan training loop lengkap dengan Keras/TensorFlow

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print("Library berhasil diimport!")

## 1. Forward Propagation - Penjelasan Matematis

Forward propagation adalah proses menghitung output dari input melewati semua layer jaringan.

Untuk layer $l$:
$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$$
$$A^{[l]} = g^{[l]}(Z^{[l]})$$

Dimana:
- $W^{[l]}$ = matriks bobot layer $l$
- $b^{[l]}$ = bias layer $l$
- $g^{[l]}$ = fungsi aktivasi layer $l$
- $A^{[0]} = X$ (input)

In [ ]:
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def forward_propagation_demo():
    """
    Demonstrasi forward propagation step-by-step
    Arsitektur: 2 -> 3 -> 1
    """
    np.random.seed(0)
    
    # Input (1 sampel, 2 fitur)
    X = np.array([[0.5, 0.8]])
    print(f"Input X: {X}")
    print(f"Shape: {X.shape}\n")
    
    # Layer 1: 2 -> 3
    W1 = np.random.randn(2, 3) * 0.1
    b1 = np.zeros((1, 3))
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    print(f"=== LAYER 1 ===")
    print(f"W1 shape: {W1.shape}")
    print(f"Z1 = X·W1 + b1 = {Z1.round(4)}")
    print(f"A1 = ReLU(Z1)  = {A1.round(4)}\n")
    
    # Layer 2 (output): 3 -> 1
    W2 = np.random.randn(3, 1) * 0.1
    b2 = np.zeros((1, 1))
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    print(f"=== LAYER 2 (OUTPUT) ===")
    print(f"W2 shape: {W2.shape}")
    print(f"Z2 = A1·W2 + b2 = {Z2.round(4)}")
    print(f"A2 = σ(Z2)      = {A2.round(4)}")
    print(f"\nPrediksi: {A2[0][0]:.4f}")

forward_propagation_demo()

## 2. Loss Functions (Fungsi Kerugian)

Loss function mengukur seberapa jauh prediksi model dari nilai sebenarnya.

### Mean Squared Error (Regresi):
$$\mathcal{L}_{MSE} = \frac{1}{m}\sum_{i=1}^{m}(y^{(i)} - \hat{y}^{(i)})^2$$

### Binary Cross-Entropy (Klasifikasi Biner):
$$\mathcal{L}_{BCE} = -\frac{1}{m}\sum_{i=1}^{m}[y^{(i)}\log\hat{y}^{(i)} + (1-y^{(i)})\log(1-\hat{y}^{(i)})]$$

### Categorical Cross-Entropy (Klasifikasi Multi-kelas):
$$\mathcal{L}_{CCE} = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{K}y_k^{(i)}\log\hat{y}_k^{(i)}$$

In [ ]:
# Visualisasi Loss Functions
y_true = 1  # Label sebenarnya
y_pred = np.linspace(0.001, 0.999, 200)

# Binary Cross-Entropy saat y=1
bce_y1 = -np.log(y_pred)  # Saat y_true=1
# Binary Cross-Entropy saat y=0
bce_y0 = -np.log(1 - y_pred)  # Saat y_true=0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Visualisasi Binary Cross-Entropy Loss', fontsize=14, fontweight='bold')

# Plot BCE saat y=1
axes[0].plot(y_pred, bce_y1, 'blue', linewidth=2)
axes[0].set_title('BCE Loss ketika y_true = 1\n(semakin kecil prediksi, semakin besar loss)')
axes[0].set_xlabel('Prediksi (ŷ)')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 7)
axes[0].annotate('Prediksi = 1.0\nLoss ≈ 0 ✓', xy=(0.95, 0.05), xytext=(0.6, 1.5),
                 arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=9)
axes[0].annotate('Prediksi = 0.0\nLoss → ∞ ✗', xy=(0.05, 3), xytext=(0.3, 4),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)

# Plot BCE saat y=0
axes[1].plot(y_pred, bce_y0, 'orange', linewidth=2)
axes[1].set_title('BCE Loss ketika y_true = 0\n(semakin besar prediksi, semakin besar loss)')
axes[1].set_xlabel('Prediksi (ŷ)')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 7)

plt.tight_layout()
plt.show()

## 3. Gradient Descent

Gradient descent adalah algoritma optimasi yang digunakan untuk meminimalkan loss function dengan mengupdate parameter ke arah negatif gradien.

$$\theta \leftarrow \theta - \alpha \cdot \nabla_{\theta} \mathcal{L}$$

Dimana:
- $\theta$ = parameter model (bobot dan bias)
- $\alpha$ = learning rate
- $\nabla_{\theta} \mathcal{L}$ = gradien loss terhadap parameter

### Jenis-jenis Gradient Descent:
1. **Batch Gradient Descent**: Menggunakan seluruh dataset
2. **Stochastic Gradient Descent (SGD)**: Menggunakan 1 sampel per iterasi
3. **Mini-batch Gradient Descent**: Menggunakan sebagian kecil dataset (paling umum)

In [ ]:
# Visualisasi Gradient Descent
def loss_function(w):
    """Fungsi loss sederhana untuk visualisasi: f(w) = w^2 + 2w + 1"""
    return w**2 + 2*w + 1

def gradient(w):
    """Gradien dari fungsi loss: f'(w) = 2w + 2"""
    return 2*w + 2

# Gradient Descent
w_history = [3.0]  # Mulai dari w=3
w = 3.0
lr = 0.2
n_steps = 15

for _ in range(n_steps):
    grad = gradient(w)
    w = w - lr * grad
    w_history.append(w)

# Visualisasi
w_range = np.linspace(-4, 4, 200)
loss_range = loss_function(w_range)

plt.figure(figsize=(10, 6))
plt.plot(w_range, loss_range, 'b-', linewidth=2, label='Loss Function')
plt.scatter(w_history, [loss_function(w) for w in w_history],
            c=range(len(w_history)), cmap='Reds', s=100, zorder=5, label='Langkah GD')

# Gambar panah untuk menunjukkan arah update
for i in range(min(8, len(w_history)-1)):
    plt.annotate('', xy=(w_history[i+1], loss_function(w_history[i+1])),
                 xytext=(w_history[i], loss_function(w_history[i])),
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

plt.axvline(x=-1, color='green', linestyle='--', alpha=0.7, label='Minimum (w=-1)')
plt.title('Gradient Descent - Menemukan Minimum Loss', fontsize=13, fontweight='bold')
plt.xlabel('Parameter (w)')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Nilai w awal: {w_history[0]:.4f}")
print(f"Nilai w akhir: {w_history[-1]:.4f} (mendekati minimum di w=-1)")

## 4. Backpropagation

Backpropagation adalah algoritma untuk menghitung gradien loss terhadap semua parameter dengan efisien menggunakan chain rule.

### Chain Rule:
$$\frac{\partial \mathcal{L}}{\partial W^{[1]}} = \frac{\partial \mathcal{L}}{\partial A^{[2]}} \cdot \frac{\partial A^{[2]}}{\partial Z^{[2]}} \cdot \frac{\partial Z^{[2]}}{\partial A^{[1]}} \cdot \frac{\partial A^{[1]}}{\partial Z^{[1]}} \cdot \frac{\partial Z^{[1]}}{\partial W^{[1]}}$$

### Langkah-langkah Backpropagation:
1. **Forward pass**: Hitung prediksi dan loss
2. **Hitung gradien output**: $\delta^{[L]} = \hat{y} - y$
3. **Propagasi ke belakang**: Hitung gradien setiap layer
4. **Update parameter**: $W \leftarrow W - \alpha \cdot \nabla_W$

## 5. Algoritma Optimasi

Berbagai optimizer yang umum digunakan dalam Deep Learning:

In [ ]:
# Perbandingan Optimizer: SGD vs Adam vs RMSprop
# Menggunakan dataset Moons sebagai contoh

# Buat dataset
np.random.seed(42)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalisasi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def create_model(optimizer_name):
    """Buat model dengan optimizer yang berbeda"""
    model = keras.Sequential([
        keras.layers.Dense(16, activation='relu', input_shape=(2,)),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    optimizers = {
        'SGD': keras.optimizers.SGD(learning_rate=0.01),
        'Adam': keras.optimizers.Adam(learning_rate=0.01),
        'RMSprop': keras.optimizers.RMSprop(learning_rate=0.01)
    }
    
    model.compile(
        optimizer=optimizers[optimizer_name],
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Melatih model dengan berbagai optimizer...")
histories = {}
optimizer_names = ['SGD', 'Adam', 'RMSprop']

for opt_name in optimizer_names:
    tf.random.set_seed(42)
    model = create_model(opt_name)
    history = model.fit(
        X_train_scaled, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    histories[opt_name] = history
    test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"{opt_name:10s}: Test Accuracy = {test_acc:.4f}")

In [ ]:
# Visualisasi perbandingan optimizer
colors = {'SGD': 'blue', 'Adam': 'red', 'RMSprop': 'green'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Perbandingan Optimizer: SGD vs Adam vs RMSprop', fontsize=13, fontweight='bold')

for opt_name, history in histories.items():
    axes[0].plot(history.history['val_loss'], color=colors[opt_name],
                 label=opt_name, linewidth=2)
    axes[1].plot(history.history['val_accuracy'], color=colors[opt_name],
                 label=opt_name, linewidth=2)

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Overfitting dan Regularisasi

### Problem Overfitting
- **Underfitting**: Model terlalu sederhana, error tinggi di training dan validation
- **Good fit**: Model tepat, error rendah di training dan validation
- **Overfitting**: Model terlalu kompleks, error rendah di training tapi tinggi di validation

### Teknik Regularisasi:
1. **L1/L2 Regularization**: Menambahkan penalti pada bobot yang besar
2. **Dropout**: Secara acak menonaktifkan neuron saat training
3. **Early Stopping**: Berhenti training saat validation loss mulai naik
4. **Batch Normalization**: Normalisasi aktivasi antar layer

In [ ]:
# Demo: Overfitting vs Regularisasi
# Buat dataset regresi kecil
np.random.seed(42)
n_samples = 50
X_reg = np.random.uniform(-3, 3, (n_samples, 1))
y_reg = np.sin(X_reg) + 0.3 * np.random.randn(n_samples, 1)

# Split
X_tr, X_val, y_tr, y_val = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

def build_regression_model(use_dropout=False, use_l2=False):
    layers = [keras.layers.Dense(64, activation='relu', input_shape=(1,))]
    if use_l2:
        layers = [keras.layers.Dense(64, activation='relu', input_shape=(1,),
                                     kernel_regularizer=keras.regularizers.l2(0.01))]
    if use_dropout:
        layers.append(keras.layers.Dropout(0.3))
    layers.append(keras.layers.Dense(64, activation='relu'))
    if use_dropout:
        layers.append(keras.layers.Dropout(0.3))
    layers.append(keras.layers.Dense(1))
    
    model = keras.Sequential(layers)
    model.compile(optimizer='adam', loss='mse')
    return model

# Train 3 model
results = {}
configs = [
    ('Tanpa Regularisasi', False, False),
    ('Dengan Dropout', True, False),
    ('Dengan L2', False, True)
]

for name, dropout, l2 in configs:
    tf.random.set_seed(42)
    model = build_regression_model(dropout, l2)
    history = model.fit(X_tr, y_tr, epochs=200, validation_data=(X_val, y_val),
                        verbose=0)
    results[name] = (model, history)
    train_loss = history.history['loss'][-1]
    val_loss = history.history['val_loss'][-1]
    print(f"{name:25s}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

In [ ]:
# Visualisasi hasil regularisasi
X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)
colors_reg = ['red', 'green', 'blue']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Efek Regularisasi pada Overfitting', fontsize=13, fontweight='bold')

# Plot prediksi
axes[0].scatter(X_tr, y_tr, alpha=0.5, label='Training data', s=30, color='gray')
axes[0].scatter(X_val, y_val, alpha=0.5, label='Validation data', s=30, color='black', marker='x')
axes[0].plot(X_plot, np.sin(X_plot), 'k--', linewidth=1.5, label='True function', alpha=0.5)

for (name, dropout, l2), color in zip(configs, colors_reg):
    model, _ = results[name]
    y_plot = model.predict(X_plot, verbose=0)
    axes[0].plot(X_plot, y_plot, color=color, linewidth=2, label=name)

axes[0].set_title('Prediksi Model')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 2)

# Plot loss curves
for (name, dropout, l2), color in zip(configs, colors_reg):
    _, history = results[name]
    axes[1].plot(history.history['val_loss'], color=color, linewidth=2, label=name)

axes[1].set_title('Validation Loss selama Training')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 7. Implementasi Lengkap: Klasifikasi dengan Keras

Sekarang mari kita implementasikan neural network lengkap dengan best practices menggunakan Keras.

In [ ]:
# Dataset: Circles (non-linearly separable)
X_circles, y_circles = make_circles(n_samples=1000, noise=0.1, factor=0.3, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_circles, y_circles, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Build model dengan best practices
tf.random.set_seed(42)
model = keras.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(2,),
                       kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dense(1, activation='sigmoid')
], name='CircleClassifier')

model.summary()

In [ ]:
# Compile dan train
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks: Early stopping dan Learning rate reduction
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
]

history = model.fit(
    X_tr_s, y_tr,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Evaluasi dan visualisasi hasil
test_loss, test_acc = model.evaluate(X_te_s, y_te, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Visualisasi training history dan decision boundary
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Hasil Training Neural Network - Dataset Circles', fontsize=13, fontweight='bold')

# Loss curve
axes[0].plot(history.history['loss'], label='Train Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='orange')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history.history['accuracy'], label='Train Acc', color='blue')
axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='orange')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Decision boundary
h = 0.02
x_min, x_max = X_te_s[:, 0].min() - 0.5, X_te_s[:, 0].max() + 0.5
y_min, y_max = X_te_s[:, 1].min() - 0.5, X_te_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()], verbose=0)
Z = Z.reshape(xx.shape)

axes[2].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
axes[2].scatter(X_te_s[y_te==0, 0], X_te_s[y_te==0, 1], c='blue', s=20, alpha=0.7, label='Kelas 0')
axes[2].scatter(X_te_s[y_te==1, 0], X_te_s[y_te==1, 1], c='red', s=20, alpha=0.7, label='Kelas 1')
axes[2].set_title(f'Decision Boundary\n(Test Acc: {test_acc:.3f})')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Latihan Mandiri

1. **Latihan 1**: Tambahkan lebih banyak layer dan neuron. Apakah model mengalami overfitting?

2. **Latihan 2**: Ubah learning rate menjadi 0.1 dan 0.0001. Bandingkan kecepatan konvergensi.

3. **Latihan 3**: Hapus BatchNormalization. Apakah training menjadi lebih sulit?

4. **Latihan 4**: Gunakan dataset `make_moons` sebagai pengganti `make_circles`.

## Ringkasan

✅ **Forward Propagation**: $Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}$, $A^{[l]} = g(Z^{[l]})$

✅ **Loss Functions**: MSE untuk regresi, Cross-Entropy untuk klasifikasi

✅ **Gradient Descent**: $\theta \leftarrow \theta - \alpha \nabla_\theta \mathcal{L}$

✅ **Backpropagation**: Menghitung gradien dengan chain rule secara efisien

✅ **Optimizer**: SGD, Adam, RMSprop - Adam umumnya pilihan terbaik

✅ **Regularisasi**: Dropout, L2, Early Stopping, Batch Normalization

---

**Pertemuan Berikutnya:** Convolutional Neural Networks (CNN) untuk Computer Vision 🖼️